# Data (re)Preprocessing — Treating outside London as an additional MSOA, National deprivation frame.

**Changes from the 20260615 version:**
- This version of data preprocessing extend flows with a "+1" MSOA for all outside London MSOAs. These flows treat London as either origin or destination.

**Everything else is identical to the 20260615 version.** 

**Purpose:**

The main analysis filters to London-internal flows only (both origin AND destination in London). This script extends the pipeline by also including flows where ONE endpoint is in London and the other is outside, treating the entire non-London area as a single synthetic "+1" MSOA.

However, previous preprocessing logic ranked MSOAs only within London. If we introduce outside London MSOAs, there is issue with their rankings. So in this file, all ~6,800 England MSOAs are ranked and assigned to deciles based on their IMD 2010 scores on a **national frame**.

Non-London areas are then collapsed into a single sythetic MSOA (`EXT_OUTSIDE`) whose decile is the population-weighted average of all non-London MSOAs' nationally-assigned deciles. This allows external flows to be classified as "wealthier inflow" or "poorer outflow" **within a consistent national hierarchy**, **testing whether including London-external flows changes the observed cascade-counter balance**.

**Design decisions:**

| Main analysis (existing) | This extension |
| :--- | :--- |
| Deciles: 983 London MSOAs only | Deciles: ~6,800 England MSOAs |
| Flows: London ↔ London | Flows: London ↔ anywhere |
| Frame: London-relative | Frame: National-relative |
| Purpose: Core analysis | Purpose: Sensitivity/extension |

**Input:**

census_od_2021_msoa.csv        (2021 MSOA-level OD, all E&W)
census_od_2011_oa.csv          (2011 OA-level OD, all E&W)
NSPCL_NOV22_UK_LU.csv          (postcode lookup: OA → LSOA → MSOA)
msoa_2011_to_2021_lookup.csv   (MSOA 2011 ↔ 2021 correspondence)
imd_2010.xls                   (IMD 2010 LSOA scores)
imd_2019.csv                   (IMD 2019, for mid-2015 LSOA populations)
ks101ew_lsoa_2011.csv          (2011 Census population by LSOA)
msoa_cascade_features_enriched_20260616.csv  (existing results to merge)

**Design:**
1. Aggregate IMD 2010 to all England MSOAs
2. Assign wealth deciles on the **national** distribution
3. Include London-external flows in the cascade computation
4. Compare results against the London-relative baseline


**Output:**

msoa_cascade_national_frame_20260622.csv

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import date
from scipy import stats
from pyprojroot import here

In [2]:
# ══════════════════════════════════════════════════════════════════════
# CONFIGURATION — update paths to match your local setup
# ══════════════════════════════════════════════════════════════════════

ROOT = here()

DATA_DIR = ROOT / 'data'
OUTPUT_DIR = ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

# Input files (same as main preprocessing)
imd_2010_path       = DATA_DIR / 'imd_2010.xls'
census_od_2021_path = DATA_DIR / 'census_od_2021_msoa.csv'
census_od_2011_path = DATA_DIR / 'census_od_2011_oa.csv'
lookup_path         = DATA_DIR / 'NSPCL_NOV22_UK_LU.csv'
msoa_lookup_path    = DATA_DIR / 'msoa_2011_to_2021_lookup.csv'
existing_path       = OUTPUT_DIR / 'msoa_cascade_features_enriched_20260616.csv'

# NEW: all-England KS101 population file
ks101_allengland_path = DATA_DIR / 'ks101ew_lsoa_2011_allengland.csv'

# Synthetic code for all non-London areas
EXTERNAL_CODE = 'EXT_OUTSIDE'

# IMD 2010 column names
IMD_2010_LSOA_COL  = 'LSOA CODE'
IMD_2010_SCORE_COL = 'IMD SCORE'

## 1. Load raw data

In [3]:
# ---- IMD 2010 (LSOA level, all England) ----
imd_2010 = pd.read_excel(imd_2010_path, sheet_name='IMD 2010')
imd_2010.columns = imd_2010.columns.str.strip()
print(f'IMD 2010: {len(imd_2010)} LSOAs')

# ---- KS101EW population (all England & Wales LSOAs) ----
ks101 = pd.read_csv(ks101_allengland_path)
# Detect LSOA code column: find column where values match E01/W01 pattern
_ks_code_col = next(
    c for c in ks101.columns
    if ks101[c].astype(str).str.match(r'^[EW]01\d{6}$').mean() > 0.9
)
_ks_pop_col = next(c for c in ks101.columns if 'all usual residents' in c.lower())
lsoa_pop = (ks101[[_ks_code_col, _ks_pop_col]]
            .rename(columns={_ks_code_col: 'lsoa11cd', _ks_pop_col: 'pop'}))
lsoa_pop['pop'] = pd.to_numeric(lsoa_pop['pop'], errors='coerce').fillna(0)
print(f'KS101EW: {len(lsoa_pop)} LSOAs, total pop = {lsoa_pop["pop"].sum():,.0f}')

# ---- NSPCL postcode lookup (LSOA → MSOA mapping) ----
nspcl = pd.read_csv(lookup_path, encoding='ISO-8859-1', low_memory=False)
lsoa_to_msoa = nspcl[['lsoa11cd', 'msoa11cd']].drop_duplicates().dropna()
oa_to_msoa = nspcl[['oa11cd', 'msoa11cd']].drop_duplicates().dropna()
oa_to_msoa_dict = dict(zip(oa_to_msoa['oa11cd'], oa_to_msoa['msoa11cd']))
print(f'LSOA→MSOA: {len(lsoa_to_msoa)} | OA→MSOA: {len(oa_to_msoa_dict):,}')

# ---- MSOA 2021→2011 correspondence ----
msoa_11_21 = pd.read_csv(msoa_lookup_path)
msoa_11_21.columns = msoa_11_21.columns.str.strip().str.lower()
unchanged = msoa_11_21[msoa_11_21['msoa11cd'] == msoa_11_21['msoa21cd']]
msoa21_to_11 = dict(zip(unchanged['msoa21cd'], unchanged['msoa11cd']))
print(f'MSOA 2021→2011 (1:1 only): {len(msoa21_to_11)}')

# ---- Existing London-only results (for comparison at the end) ----
existing = pd.read_csv(existing_path)
london_msoas = set(existing['msoa11cd'].unique())
print(f'London analysis MSOAs: {len(london_msoas)}')

IMD 2010: 32482 LSOAs
KS101EW: 34753 LSOAs, total pop = 56,075,912
LSOA→MSOA: 42621 | OA→MSOA: 232,044
MSOA 2021→2011 (1:1 only): 7080
London analysis MSOAs: 983


## 2. National IMD aggregation: LSOA -> MSOA (all England)

Aggregate IMD 2010 to ALL England MSOAs, then qcut into deciles

Same logic as the main preprocessing:
- Population-weighted mean of LSOA IMD scores per MSOA
- Re-rank MSOAs on the aggregated scores
- Assign deciles via qcut on the national distribution

In [4]:
# ---- 2a. Join IMD scores → MSOA geography → population weights ----
imd_lsoa = imd_2010[[IMD_2010_LSOA_COL, IMD_2010_SCORE_COL]].rename(
    columns={IMD_2010_LSOA_COL: 'lsoa11cd', IMD_2010_SCORE_COL: 'imd_score'})

# LSOA → MSOA
imd_with_msoa = pd.merge(imd_lsoa, lsoa_to_msoa, on='lsoa11cd', how='inner')
print(f'IMD LSOAs matched to MSOA: {len(imd_with_msoa)} / {len(imd_lsoa)}')

# Add population weights
imd_with_msoa = pd.merge(imd_with_msoa, lsoa_pop, on='lsoa11cd', how='left')
imd_with_msoa['pop'] = imd_with_msoa['pop'].fillna(0)

IMD LSOAs matched to MSOA: 31672 / 32482


In [5]:
# ---- 2b. Population-weighted mean per MSOA ----
# (Same fallback as main notebook: if all weights are zero, use simple mean)
def pop_weighted_mean(group):
    total_pop = group['pop'].sum()
    if total_pop > 0:
        return np.average(group['imd_score'], weights=group['pop'])
    else:
        return group['imd_score'].mean()

msoa_imd = (imd_with_msoa
            .groupby('msoa11cd')
            .apply(pop_weighted_mean, include_groups=False)
            .reset_index(name='IMD_2010_national'))

print(f'MSOAs with aggregated IMD: {len(msoa_imd)}')
print(f'  London MSOAs:     {msoa_imd["msoa11cd"].isin(london_msoas).sum()}')
print(f'  Non-London MSOAs: {(~msoa_imd["msoa11cd"].isin(london_msoas)).sum()}')


MSOAs with aggregated IMD: 6778
  London MSOAs:     983
  Non-London MSOAs: 5795


In [6]:
# ---- 2c. Assign national wealth deciles ----
# qcut splits into 10 bins by score. Higher IMD score = more deprived.
# pd.qcut with labels=False gives bin 0 = lowest scores = LEAST deprived.
# We want Decile 1 = most deprived, so: Decile = 11 - (bin + 1)
msoa_imd['Wealth_Decile_National'] = (
    11 - (pd.qcut(msoa_imd['IMD_2010_national'], 10, labels=False) + 1)
)

# Verify direction: D1 should have highest IMD scores (most deprived)
d1_score = msoa_imd.loc[msoa_imd['Wealth_Decile_National'] == 1,
                         'IMD_2010_national'].mean()
d10_score = msoa_imd.loc[msoa_imd['Wealth_Decile_National'] == 10,
                          'IMD_2010_national'].mean()
assert d1_score > d10_score, \
    f'Decile direction wrong: D1 mean={d1_score:.1f}, D10 mean={d10_score:.1f}'
print(f'\nDecile direction verified: D1 (most deprived, mean={d1_score:.1f}) > '
      f'D10 (least deprived, mean={d10_score:.1f})')



Decile direction verified: D1 (most deprived, mean=50.7) > D10 (least deprived, mean=5.7)


In [7]:
# ---- 2d. Summary: national decile composition ----
print(f'\nNational decile distribution:')
print(f'{"Decile":>6s} {"N_total":>8s} {"N_London":>9s} {"Mean IMD Score":>9s}')
for d in range(1, 11):
    mask = msoa_imd['Wealth_Decile_National'] == d
    n_total = mask.sum()
    n_london = (mask & msoa_imd['msoa11cd'].isin(london_msoas)).sum()
    mean_imd = msoa_imd.loc[mask, 'IMD_2010_national'].mean()
    print(f'  D{d:>2d}   {n_total:>7d}  {n_london:>8d}  {mean_imd:>9.2f}')



National decile distribution:
Decile  N_total  N_London Mean IMD Score
  D 1       678       107      50.73
  D 2       678       182      36.99
  D 3       678       149      29.63
  D 4       677       135      24.06
  D 5       678       107      19.72
  D 6       678        76      16.22
  D 7       677        74      13.41
  D 8       678        65      11.03
  D 9       678        52       8.74
  D10       678        36       5.73


> **Not the heavy tails on both ends as we expected? Specifically, there're more deprived MSOAs in London?**

In [8]:
# ---- 2e. External MSOA decile ----
# Population-weighted mean decile of all non-London MSOAs
non_london_imd = msoa_imd[~msoa_imd['msoa11cd'].isin(london_msoas)].copy()

# Get MSOA-level population for weighting
msoa_pop = (imd_with_msoa.groupby('msoa11cd')['pop'].sum()
            .reset_index(name='msoa_pop'))
non_london_imd = non_london_imd.merge(msoa_pop, on='msoa11cd', how='left')
non_london_imd['msoa_pop'] = non_london_imd['msoa_pop'].fillna(0)

# Safety check
total_ext_pop = non_london_imd['msoa_pop'].sum()
print(f'\nExternal MSOA calculation:')
print(f'  Non-London MSOAs: {len(non_london_imd)}, total pop: {total_ext_pop:,.0f}')
assert total_ext_pop > 0, 'External population is zero — check KS101 file coverage'

ext_weighted_decile = np.average(
    non_london_imd['Wealth_Decile_National'],
    weights=non_london_imd['msoa_pop'])
ext_decile = int(round(ext_weighted_decile))
print(f'  Weighted mean decile: {ext_weighted_decile:.2f} → assigned D{ext_decile}')


External MSOA calculation:
  Non-London MSOAs: 5795, total pop: 43,223,548
  Weighted mean decile: 5.67 → assigned D6


> **MSOAs of external flows are mainly in D6.**

In [9]:
# ---- 2f. Build wealth mapping (all MSOAs + external) ----
wealth_national = dict(zip(msoa_imd['msoa11cd'],
                            msoa_imd['Wealth_Decile_National']))
wealth_national[EXTERNAL_CODE] = ext_decile

In [10]:
# ---- 2g. Compare national vs London-relative deciles ----
print(f'\nLondon MSOA decile comparison (London-relative vs National):')
london_comparison = msoa_imd[msoa_imd['msoa11cd'].isin(london_msoas)].merge(
    existing[['msoa11cd', 'Wealth_Decile']], on='msoa11cd')

ct = pd.crosstab(london_comparison['Wealth_Decile'],
                 london_comparison['Wealth_Decile_National'], margins=True)
print(ct.to_string())

rho, p = stats.spearmanr(london_comparison['Wealth_Decile'],
                          london_comparison['Wealth_Decile_National'])
print(f'\nSpearman ρ = {rho:.4f}, p = {p:.2e}')


London MSOA decile comparison (London-relative vs National):
Wealth_Decile_National    1    2    3    4    5   6   7   8   9  10  All
Wealth_Decile                                                           
1                        99    0    0    0    0   0   0   0   0   0   99
2                         8   90    0    0    0   0   0   0   0   0   98
3                         0   92    6    0    0   0   0   0   0   0   98
4                         0    0   98    0    0   0   0   0   0   0   98
5                         0    0   45   53    0   0   0   0   0   0   98
6                         0    0    0   82   17   0   0   0   0   0   99
7                         0    0    0    0   90   8   0   0   0   0   98
8                         0    0    0    0    0  68  30   0   0   0   98
9                         0    0    0    0    0   0  44  54   0   0   98
10                        0    0    0    0    0   0   0  11  52  36   99
All                     107  182  149  135  107  76  74  65  5

### Interpretation

Rows are existing London-relative deciles.
Columns are national deciles.

- The striking pattern is the downward shift.
    - In the very bottom line, 107 MSOAs in D1, but only 36 in D10.
    - **London is more deprived than England on average. When ranking London MSOAs against the whole country, they pile up in the lower deciles.**

- London D1 all stay at national D1.
    -  These are the most deprived in the country, not just in London.

- London D5 splits into national D3 and national D4.
    - "Middle" within London is actually below-average nationally.

- London D10 only have 36 of 99 in national D10 as well.
    - The rest London D10 spread across national D7 and D9.
    - Even London's wealthiest neighbourhoods are not all in the national top 10%.

- rho = 0.99 means the ordering is almost identical.
    - e.g. If MSOA a is wealthier than MSOA b within London, then it should be wealthier natioanlly.
    - Even the fact of "wealthier" holding, the levels would compress downward.
    - **The internal hierarchy is preserved, but only the absolute position changes.**

External MSOAs are mostly at D6 nationally, sitting above roughly 75% of London MSOAs in national frame, since only 164 London MSOAs reach national D6 or higher.

So, **most flows from outside London into London would register as `Inflow_Wealthier` (cascade direction), and most flows from London to outside will register as `Outflow_Wealthier`.**

## 3. Filter OD data: London-touching flows

Keep flows where at least one endpoint is a Lonson MSOA.
Non-London endpoints are recoded as `EXTERNAL_CODE`.
We exclude intra-MSOA moves.

In [20]:
# ---- 3a. 2021 Census OD (MSOA-level, 2021 codes) ----
print('\n--- 2021 ---')
ORIGIN_COL_2021 = 'Migrant MSOA one year ago code'
DEST_COL_2021   = 'Middle layer Super Output Areas code'
 
od_2021 = pd.read_csv(census_od_2021_path)
od_2021[ORIGIN_COL_2021] = od_2021[ORIGIN_COL_2021].astype(str).str.strip()
od_2021[DEST_COL_2021]   = od_2021[DEST_COL_2021].astype(str).str.strip()
 
# Detect count column
count_cols = [c for c in od_2021.columns
              if 'observation' in c.lower() or 'count' in c.lower()]
COUNT_COL_2021 = count_cols[0] if count_cols else od_2021.columns[-1]
print(f'Count column: {COUNT_COL_2021!r}')
print(f'Raw records: {len(od_2021):,}')


--- 2021 ---
Count column: 'Count'
Raw records: 1,490,726


In [21]:
# ── Step 1: Remove non-geography pseudo-codes ──────────────────────
# -8 = "Does not apply" (non-migrants); 999999999 = no fixed place;
# N99... = ONS "elsewhere" pseudo-codes.
# Without this, -8 rows (~53M non-migrant residents) get recoded as
# EXT_OUTSIDE inflows downstream.
# Ref: London-only notebook Section 4, classify_code_2021()
SPECIAL_2021 = {'-8', '999999999'}
 
is_special_origin = (od_2021[ORIGIN_COL_2021].isin(SPECIAL_2021) |
                     od_2021[ORIGIN_COL_2021].str.startswith('N99', na=False))
is_special_dest   = (od_2021[DEST_COL_2021].isin(SPECIAL_2021) |
                     od_2021[DEST_COL_2021].str.startswith('N99', na=False))
special_mask = is_special_origin | is_special_dest
 
print(f'Special-code rows removed:  {special_mask.sum():>10,} records  '
      f'{od_2021.loc[special_mask, COUNT_COL_2021].sum():>12,.0f} persons')
print(f'  of which -8 origin:       {is_special_origin.sum():>10,} records  '
      f'{od_2021.loc[is_special_origin, COUNT_COL_2021].sum():>12,.0f} persons')
 
od_2021 = od_2021[~special_mask].copy()
print(f'Records after filter: {len(od_2021):,}')

Special-code rows removed:      16,729 records    53,046,196 persons
  of which -8 origin:           16,729 records    53,046,196 persons
Records after filter: 1,473,997


In [22]:
# ── Step 2: Harmonise 2021→2011 codes ──────────────────────────────
od_2021['origin_msoa11'] = od_2021[ORIGIN_COL_2021].map(msoa21_to_11)
od_2021['dest_msoa11']   = od_2021[DEST_COL_2021].map(msoa21_to_11)

In [23]:
# ── Step 3: Classify London vs non-London ──────────────────────────
od_2021['o_ldn'] = od_2021['origin_msoa11'].isin(london_msoas)
od_2021['d_ldn'] = od_2021['dest_msoa11'].isin(london_msoas)

In [24]:
# Keep: at least one endpoint in London
lt_21 = od_2021[od_2021['o_ldn'] | od_2021['d_ldn']].copy()

In [25]:
# ── Step 4: Flow-weighted external decile diagnostic ───────────────
# Compute BEFORE collapsing non-London MSOAs, while individual codes
# are still available. Checks whether using a single D6 for
# EXT_OUTSIDE is a reasonable simplification.
ext_in_21  = lt_21[~lt_21['o_ldn'] & lt_21['d_ldn']].copy()
ext_out_21 = lt_21[lt_21['o_ldn'] & ~lt_21['d_ldn']].copy()
 
ext_in_21['o_dec']  = ext_in_21['origin_msoa11'].map(wealth_national)
ext_out_21['d_dec'] = ext_out_21['dest_msoa11'].map(wealth_national)
 
ext_in_21  = ext_in_21.dropna(subset=['o_dec'])
ext_out_21 = ext_out_21.dropna(subset=['d_dec'])
 
fw_in_21 = (np.average(ext_in_21['o_dec'], weights=ext_in_21[COUNT_COL_2021])
            if len(ext_in_21) > 0 and ext_in_21[COUNT_COL_2021].sum() > 0
            else np.nan)
fw_out_21 = (np.average(ext_out_21['d_dec'], weights=ext_out_21[COUNT_COL_2021])
             if len(ext_out_21) > 0 and ext_out_21[COUNT_COL_2021].sum() > 0
             else np.nan)
 
print(f'\n  Flow-weighted external decile (2021):')
print(f'    Population-weighted (all non-Ldn):  {ext_weighted_decile:.2f} → D{ext_decile}')
print(f'    Inflow origins  (Ext→Ldn):          {fw_in_21:.2f} → D{int(round(fw_in_21))}  '
      f'({ext_in_21[COUNT_COL_2021].sum():,.0f} persons)')
print(f'    Outflow dests   (Ldn→Ext):          {fw_out_21:.2f} → D{int(round(fw_out_21))}  '
      f'({ext_out_21[COUNT_COL_2021].sum():,.0f} persons)')


  Flow-weighted external decile (2021):
    Population-weighted (all non-Ldn):  5.67 → D6
    Inflow origins  (Ext→Ldn):          6.36 → D6  (130,057 persons)
    Outflow dests   (Ldn→Ext):          6.47 → D6  (298,864 persons)


### Interpretation:

People who moved to London in 2021 came from deciles wealthier than the average non-London area. And people also leaft to splightly wealthier areas. Both around D6.
> Why would comparing with the average London deciles matter? Does this imply anything?
- Population-weighted mean across all 5795 non-London MSOAs is 5.67
- Inflow origins average decile is 6.36, 0.7 wealthier.
- Outflow destination average decile is 6.47.

**In 2021, London's migration exchange is with the more affluent parts of England, rather than with the national average.**

**In 2021, outflow destinations being slightly wealthier than inflow origins (6.47 - 6.36) is consistent with the escalator model, where people leaving London tend to "cash out" to wealthier areas.**

> Would the assigned decile of outside London MSOAs differ by year?

Given we only use 2010 IMD to classify MSOA deciles, we can ignore the temporal changes between 2010 IMD and 2019 IMD. The more important is **the composition change of actual migrants between London and the rest of England**.

In reality, London's in-migrant probably come disproportionately from specific types of areas, which are not uniformly from all England. And that composition could shift between census years, especially with COVID reshaping 2021 migration patterns.

### Need to stressed in methodology:

We used 2 different ways of answering the question "what decile should we assign to `EXTERNAL_CODE`?"

**From section2, every MSOA in the data has a national-based decile. The below 2 methods evalute how to aggregate all outside London MSOAs' deciles into one.**

- **Population-weighted (used and kept)**

  Take all 5795 MSOAs and average their deciles weighted by how many people live in each one. The result 5.67 assigned as D6.

  This characterises the non-London population as a whole, and asking "if you pick a random person living outside London, what decile would they be in on average?"

  **The typical non-London resident lives in a D6-equivalent area.**

- **Flow-weighted (bottom 2 line test results)**
 
  Take only MSOAs that actually sent migrants to London and averages their deciles weighted by how many migrants each one sent Results is 6.36, assigned as D6 again.

  This checked if the population-weighted method is misleading. For example, if migrants came overwhelmingly from D9 areas, then calling external flows "D6" would systematically misclassify them.

**Reasons we need this comparison: They could have different results because migrants are not a random sample of the non-London population. As we discussed in the interpretation in the above cell, London's migration exchange is concerned with certain types of places, not uniformly with all of England.**

**But either method gives the same decile assignment as D6.**

In [26]:
# ── Step 5: Collapse non-London → EXTERNAL ─────────────────────────
lt_21.loc[~lt_21['o_ldn'], 'origin_msoa11'] = EXTERNAL_CODE
lt_21.loc[~lt_21['d_ldn'], 'dest_msoa11']   = EXTERNAL_CODE
 
# Drop intra-MSOA
lt_21 = lt_21[lt_21['origin_msoa11'] != lt_21['dest_msoa11']]

In [27]:
# ── Step 6: Aggregate ──────────────────────────────────────────────
flows_21 = (lt_21.groupby(['origin_msoa11', 'dest_msoa11'])[COUNT_COL_2021]
            .sum().reset_index().rename(columns={COUNT_COL_2021: 'count'}))
 
_i = flows_21[(flows_21['origin_msoa11'] != EXTERNAL_CODE) &
              (flows_21['dest_msoa11'] != EXTERNAL_CODE)]
_f = flows_21[flows_21['origin_msoa11'] == EXTERNAL_CODE]
_t = flows_21[flows_21['dest_msoa11'] == EXTERNAL_CODE]
print(f'\nInternal:  {len(_i):>7,} records  {_i["count"].sum():>10,.0f} persons')
print(f'Ext→Ldn:   {len(_f):>7,} records  {_f["count"].sum():>10,.0f} persons')
print(f'Ldn→Ext:   {len(_t):>7,} records  {_t["count"].sum():>10,.0f} persons')


Internal:  173,311 records     695,667 persons
Ext→Ldn:       963 records     177,759 persons
Ldn→Ext:       963 records     353,975 persons


In [28]:
# ---- 3b. 2011 Census OD (OA-level) ----
print('\n--- 2011 ---')
od_2011 = pd.read_csv(
    census_od_2011_path, header=None,
    names=['dest_oa', 'origin_oa', 'persons'],       # A=dest, B=origin, C=count
    dtype={'dest_oa': str, 'origin_oa': str, 'persons': int})
od_2011['origin_oa'] = od_2011['origin_oa'].str.strip()
od_2011['dest_oa']   = od_2011['dest_oa'].str.strip()
print(f'Raw OA records: {len(od_2011):,}')


--- 2011 ---
Raw OA records: 3,709,939


In [29]:
# ── Step 1: Remove OD-prefixed special codes ───────────────────────
# Parallel to 2021 filter. OD-prefixed rows are cross-border and
# no-fixed-origin pseudo-OAs. Small counts, but keeps logic clean.
# Ref: London-only notebook Section 4, classify_code_2011_oa()
is_special_2011 = (od_2011['origin_oa'].str.startswith('OD', na=False) |
                   od_2011['dest_oa'].str.startswith('OD', na=False))
print(f'Special-code rows removed:  {is_special_2011.sum():>10,} records  '
      f'{od_2011.loc[is_special_2011, "persons"].sum():>12,.0f} persons')
od_2011 = od_2011[~is_special_2011].copy()

Special-code rows removed:     116,320 records       612,488 persons


In [30]:
# ── Step 2: OA → MSOA ─────────────────────────────────────────────
od_2011['origin_msoa11'] = od_2011['origin_oa'].map(oa_to_msoa_dict)
od_2011['dest_msoa11']   = od_2011['dest_oa'].map(oa_to_msoa_dict)

In [31]:
# ── Step 3: Classify London vs non-London ──────────────────────────
od_2011['o_ldn'] = od_2011['origin_msoa11'].isin(london_msoas)
od_2011['d_ldn'] = od_2011['dest_msoa11'].isin(london_msoas)

In [32]:
# Keep: at least one endpoint in London
lt_11 = od_2011[od_2011['o_ldn'] | od_2011['d_ldn']].copy()

In [33]:
# ── Step 4: Flow-weighted external decile diagnostic ───────────────
ext_in_11  = lt_11[~lt_11['o_ldn'] & lt_11['d_ldn']].copy()
ext_out_11 = lt_11[lt_11['o_ldn'] & ~lt_11['d_ldn']].copy()
 
ext_in_11['o_dec']  = ext_in_11['origin_msoa11'].map(wealth_national)
ext_out_11['d_dec'] = ext_out_11['dest_msoa11'].map(wealth_national)
 
ext_in_11  = ext_in_11.dropna(subset=['o_dec'])
ext_out_11 = ext_out_11.dropna(subset=['d_dec'])
 
fw_in_11 = (np.average(ext_in_11['o_dec'], weights=ext_in_11['persons'])
            if len(ext_in_11) > 0 and ext_in_11['persons'].sum() > 0
            else np.nan)
fw_out_11 = (np.average(ext_out_11['d_dec'], weights=ext_out_11['persons'])
             if len(ext_out_11) > 0 and ext_out_11['persons'].sum() > 0
             else np.nan)
 
print(f'\n  Flow-weighted external decile (2011):')
print(f'    Population-weighted (all non-Ldn):  {ext_weighted_decile:.2f} → D{ext_decile}')
print(f'    Inflow origins  (Ext→Ldn):          {fw_in_11:.2f} → D{int(round(fw_in_11))}  '
      f'({ext_in_11["persons"].sum():,.0f} persons)')
print(f'    Outflow dests   (Ldn→Ext):          {fw_out_11:.2f} → D{int(round(fw_out_11))}  '
      f'({ext_out_11["persons"].sum():,.0f} persons)')


  Flow-weighted external decile (2011):
    Population-weighted (all non-Ldn):  5.67 → D6
    Inflow origins  (Ext→Ldn):          6.37 → D6  (170,474 persons)
    Outflow dests   (Ldn→Ext):          6.39 → D6  (212,248 persons)


In [34]:
# ── Step 5: Collapse non-London → EXTERNAL ─────────────────────────
lt_11.loc[~lt_11['o_ldn'], 'origin_msoa11'] = EXTERNAL_CODE
lt_11.loc[~lt_11['d_ldn'], 'dest_msoa11']   = EXTERNAL_CODE
 
# Drop intra-MSOA
lt_11 = lt_11[lt_11['origin_msoa11'] != lt_11['dest_msoa11']]

In [35]:
# ── Step 6: Aggregate to MSOA level ────────────────────────────────
flows_11 = (lt_11.groupby(['origin_msoa11', 'dest_msoa11'])['persons']
            .sum().reset_index().rename(columns={'persons': 'count'}))
 
_i = flows_11[(flows_11['origin_msoa11'] != EXTERNAL_CODE) &
              (flows_11['dest_msoa11'] != EXTERNAL_CODE)]
_f = flows_11[flows_11['origin_msoa11'] == EXTERNAL_CODE]
_t = flows_11[flows_11['dest_msoa11'] == EXTERNAL_CODE]
print(f'\nInternal:  {len(_i):>7,} records  {_i["count"].sum():>10,.0f} persons')
print(f'Ext→Ldn:   {len(_f):>7,} records  {_f["count"].sum():>10,.0f} persons')
print(f'Ldn→Ext:   {len(_t):>7,} records  {_t["count"].sum():>10,.0f} persons')


Internal:  173,595 records     768,423 persons
Ext→Ldn:       983 records     187,595 persons
Ldn→Ext:       983 records     219,221 persons


In [36]:
# ── Summary: flow-weighted vs population-weighted ──────────────────
print('\n' + '=' * 60)
print('EXTERNAL DECILE SUMMARY')
print('=' * 60)
print(f'  Population-weighted mean (all non-London):  D{ext_decile} ({ext_weighted_decile:.2f})')
print(f'  2011 inflow origins:   D{int(round(fw_in_11))} ({fw_in_11:.2f})   '
      f'diff from pop: {fw_in_11 - ext_weighted_decile:+.2f}')
print(f'  2011 outflow dests:    D{int(round(fw_out_11))} ({fw_out_11:.2f})   '
      f'diff from pop: {fw_out_11 - ext_weighted_decile:+.2f}')
print(f'  2021 inflow origins:   D{int(round(fw_in_21))} ({fw_in_21:.2f})   '
      f'diff from pop: {fw_in_21 - ext_weighted_decile:+.2f}')
print(f'  2021 outflow dests:    D{int(round(fw_out_21))} ({fw_out_21:.2f})   '
      f'diff from pop: {fw_out_21 - ext_weighted_decile:+.2f}')


EXTERNAL DECILE SUMMARY
  Population-weighted mean (all non-London):  D6 (5.67)
  2011 inflow origins:   D6 (6.37)   diff from pop: +0.70
  2011 outflow dests:    D6 (6.39)   diff from pop: +0.72
  2021 inflow origins:   D6 (6.36)   diff from pop: +0.69
  2021 outflow dests:    D6 (6.47)   diff from pop: +0.80


### Summary Interpretation:

The external synthetic MSOA was assigned a national wealth decile of D6 based on population-weighted mean of all non-London MSOAs.

Flow-weighted diagnostic confirmed this assignment with all values around to D6, validating the single-decile simplication across both census periods.

- Single D6 assignment is validated,
- The type of place London exchanges migrants with didn't meaningfully change between 2011 and 2021, even though the volume changes substantially. The structural geography of London's migration partnerships is durable.
- London exchanges with slightly wealthier-than-average England, not the "average England". (all 4 flow-weighted deciles sit ~0.7 deciles above the population-weighted mean)

## 4. Compute cascade metrics (national deciles, with external flows)

In [37]:
def compute_cascade(flow_df, wealth_map, year_label):
    """
    Per-London-MSOA cascade & counter-cascade metrics.
    Identical formulas to the main preprocessing, just different
    wealth_map (national deciles) and flow table (includes external).
    """
    df = flow_df.copy()
    df['o_dec'] = df['origin_msoa11'].map(wealth_map)
    df['d_dec'] = df['dest_msoa11'].map(wealth_map)

    # Drop unmapped
    before = len(df)
    df = df.dropna(subset=['o_dec', 'd_dec'])
    dropped = before - len(df)
    if dropped:
        print(f'  ⚠ {dropped} records dropped (unmapped decile)')
    df['o_dec'] = df['o_dec'].astype(int)
    df['d_dec'] = df['d_dec'].astype(int)

    # Summary
    '''
    `up` means destination decile > origin decile
    `down` means destination decile < origin decile
    `lat` means destination decile = origin decile
    '''
    total = df['count'].sum()
    up   = df.loc[df['d_dec'] > df['o_dec'], 'count'].sum()
    down = df.loc[df['d_dec'] < df['o_dec'], 'count'].sum()
    lat  = df.loc[df['d_dec'] == df['o_dec'], 'count'].sum()
    print(f'\n  {year_label}:')
    print(f'    Total:   {total:>10,.0f}')
    print(f'    Upward:  {up:>10,.0f} ({up/total*100:.1f}%)')
    print(f'    Down:    {down:>10,.0f} ({down/total*100:.1f}%)')
    print(f'    Lateral: {lat:>10,.0f} ({lat/total*100:.1f}%)')

    # ---- Per-London-MSOA metrics ----
    idx = sorted(london_msoas)

    # Cascade: wealthier in, poorer out
    inflow_w  = (df[df['o_dec'] > df['d_dec']].groupby('dest_msoa11')['count']
                 .sum().reindex(idx, fill_value=0))
    outflow_p = (df[df['d_dec'] < df['o_dec']].groupby('origin_msoa11')['count']
                 .sum().reindex(idx, fill_value=0))

    # Counter-cascade: wealthier out, poorer in
    outflow_w = (df[df['d_dec'] > df['o_dec']].groupby('origin_msoa11')['count']
                 .sum().reindex(idx, fill_value=0))
    inflow_p  = (df[df['o_dec'] < df['d_dec']].groupby('dest_msoa11')['count']
                 .sum().reindex(idx, fill_value=0))

    # Totals
    total_in  = df.groupby('dest_msoa11')['count'].sum().reindex(idx, fill_value=0)
    total_out = df.groupby('origin_msoa11')['count'].sum().reindex(idx, fill_value=0)

    # External-specific
    ext_in  = (df[df['origin_msoa11'] == EXTERNAL_CODE]
               .groupby('dest_msoa11')['count'].sum().reindex(idx, fill_value=0))
    ext_out = (df[df['dest_msoa11'] == EXTERNAL_CODE]
               .groupby('origin_msoa11')['count'].sum().reindex(idx, fill_value=0))

    # Assemble
    r = pd.DataFrame({'msoa11cd': idx})
    r = r.set_index('msoa11cd')
    r['Inflow_Wealthier']  = inflow_w.values
    r['Outflow_Poorer']    = outflow_p.values
    r['Outflow_Wealthier'] = outflow_w.values
    r['Inflow_Poorer']     = inflow_p.values
    r['Total_Inflow']      = total_in.values
    r['Total_Outflow']     = total_out.values
    r['Ext_Inflow']        = ext_in.values
    r['Ext_Outflow']       = ext_out.values

    # Derived metrics (same formulas as main notebook)
    r['Total_Migration'] = r['Total_Inflow'] + r['Total_Outflow']
    r['CFI_Churn']   = r['Inflow_Wealthier'] + r['Outflow_Poorer']
    r['CFI_Rate']    = np.where(r['Total_Migration'] > 0,
        (r['Inflow_Wealthier'] * r['Outflow_Poorer']) / r['Total_Migration'], 0)
    r['Net_Cascade'] = r['Inflow_Wealthier'] - r['Outflow_Poorer']
    r['Pct_Inflow_Wealthier'] = np.where(r['Total_Inflow'] > 0,
        r['Inflow_Wealthier'] / r['Total_Inflow'] * 100, 0)

    r['Counter_Churn'] = r['Outflow_Wealthier'] + r['Inflow_Poorer']
    r['Counter_Rate']  = np.where(r['Total_Migration'] > 0,
        (r['Outflow_Wealthier'] * r['Inflow_Poorer']) / r['Total_Migration'], 0)
    r['Net_Counter']   = r['Outflow_Wealthier'] - r['Inflow_Poorer']

    # Cascade Dominance
    total_cross = r['CFI_Churn'] + r['Counter_Churn']
    r['Cascade_Dominance'] = np.where(total_cross > 0,
        r['CFI_Churn'] / total_cross, 0.5)

    r['Ext_Net'] = r['Ext_Inflow'] - r['Ext_Outflow']
    return r


In [38]:
# Compute both periods
print('\n--- 2011 ---')
nat_11 = compute_cascade(flows_11, wealth_national, '2011 national frame')
print('\n--- 2021 ---')
nat_21 = compute_cascade(flows_21, wealth_national, '2021 national frame')

# Add _nat suffix + year suffix
nat_11.columns = [f'{c}_nat_11' for c in nat_11.columns]
nat_21.columns = [f'{c}_nat_21' for c in nat_21.columns]


--- 2011 ---

  2011 national frame:
    Total:    1,175,239
    Upward:     493,120 (42.0%)
    Down:       487,346 (41.5%)
    Lateral:    194,773 (16.6%)

--- 2021 ---

  2021 national frame:
    Total:    1,227,401
    Upward:     574,573 (46.8%)
    Down:       472,497 (38.5%)
    Lateral:    180,331 (14.7%)


## 5. Merge and Compare London-only vs. National frame

In [39]:
# Add national decile
nat_deciles = msoa_imd[msoa_imd['msoa11cd'].isin(london_msoas)][
    ['msoa11cd', 'Wealth_Decile_National']]

result = existing.merge(nat_deciles, on='msoa11cd', how='left')
result = result.merge(nat_11, left_on='msoa11cd', right_index=True, how='left')
result = result.merge(nat_21, left_on='msoa11cd', right_index=True, how='left')
print(f'Final shape: {result.shape}')

Final shape: (983, 98)


In [40]:
# ---- Cascade Dominance comparison ----
print('\n' + '-' * 50)
print('CASCADE DOMINANCE COMPARISON')
print('-' * 50)

for yr in ['11', '21']:
    dom_ldn = f'Cascade_Dominance_{yr}'
    dom_nat = f'Cascade_Dominance_nat_{yr}'

    if dom_ldn in result.columns and dom_nat in result.columns:
        ldn_mean   = result[dom_ldn].mean()
        nat_mean   = result[dom_nat].mean()
        ldn_below  = (result[dom_ldn] < 0.5).mean() * 100
        nat_below  = (result[dom_nat] < 0.5).mean() * 100

        print(f'\n20{yr}:')
        print(f'  London-only:   mean = {ldn_mean:.4f}  '
              f'({ldn_below:.1f}% of MSOAs < 0.5)')
        print(f'  National+ext:  mean = {nat_mean:.4f}  '
              f'({nat_below:.1f}% of MSOAs < 0.5)')
        print(f'  Shift:         {nat_mean - ldn_mean:+.4f}')

        rho, p = stats.spearmanr(result[dom_ldn], result[dom_nat])
        print(f'  Spearman ρ:    {rho:.4f} (p={p:.2e})')



--------------------------------------------------
CASCADE DOMINANCE COMPARISON
--------------------------------------------------

2011:
  London-only:   mean = 0.4798  (67.5% of MSOAs < 0.5)
  National+ext:  mean = 0.4863  (59.1% of MSOAs < 0.5)
  Shift:         +0.0066
  Spearman ρ:    0.5506 (p=5.14e-79)

2021:
  London-only:   mean = 0.4638  (79.1% of MSOAs < 0.5)
  National+ext:  mean = 0.4555  (71.9% of MSOAs < 0.5)
  Shift:         -0.0083
  Spearman ρ:    0.3108 (p=1.87e-23)


### Interpretation:

In section 4, **there are more upward movers than downward in both periods**, after summing all cross-decile person-moves across all 983 MSOAs.

Also for `Cascade_Dominance`, **it shifts further toward counter-cascade natioanlly in 2021 (0.486 --> 0.455).**
- The upward-mobility surplus is increasingly concentrated in a subset of MSOAs.
- The typical MSOA experiences even stronger counter-cascade flows than before.

> Where are those upward-mobility surplus MSOAs?

**In 2011:**
- Adding external flows nudges dominance toward cascade (+0.007).
    - The external MSOA sits at national D6, and ~69% of London MSOAs fall below D6 natioanlly. For those ~69% MSOAs in London, external inflows are exactly `Inflow_Wealthier`, and external outflows are `Outflow_Wealthier`.
    - The cascade-direction nudge in 2011 means the wealthier-inflow effect marginally outweighted the wealthier-outflow effect at the per-MSOA level.
- Although, London was an net exporter in a raw volume, because inflow was more evenly distributed across MSOAs while the outflow was concentrated.

**In 2021:**
- The cascade dominance shift reverses, further toward counter-cascade.
    - London's external outflow nearly doubled, while inflow slightly declined.
    - This can attribute to "fight from London" during COVID, disproportionately sent residents to wealthier areas outside London, which for ~69% of London MSOAs below national D6. Those flows are `Outflow_Wealthier`.
- Exxternal flows in 2021 reinforced the counter-cascade dominance rather than offsetting it.

**Conclusion:**
**London's boundary is not a neutral container for the cascade/counter balance, and the direction of its effect changed between census periods.**
- External flows modestly offset London's internal counter dominance in 2011, but they amplifies it by 2021.
- The pandemic-era outflow pattern turned London's external boundary from a mild cascade-supportive force into a counter-cascade-reinforcing one.

**London-only analysis understates the counter-cascade shift in 2021. The core narrative (counter-cascade dominance intensifying between censuses, cascade as a localised perturbantion) holds even more firmly once accounting for boundary effects.**

In [41]:
# ---- CFI Churn comparison ----
print('\n' + '-' * 50)
print('CFI CHURN COMPARISON')
print('-' * 50)
for yr in ['11', '21']:
    c_ldn = f'CFI_Churn_{yr}'
    c_nat = f'CFI_Churn_nat_{yr}'
    if c_ldn in result.columns and c_nat in result.columns:
        print(f'\n20{yr}:')
        print(f'  London-only mean: {result[c_ldn].mean():>10.1f}')
        print(f'  National mean:    {result[c_nat].mean():>10.1f}')
        print(f'  Increase:         {result[c_nat].mean() - result[c_ldn].mean():>+10.1f}')



--------------------------------------------------
CFI CHURN COMPARISON
--------------------------------------------------

2011:
  London-only mean:      629.9
  National mean:         797.0
  Increase:             +167.1

2021:
  London-only mean:      554.4
  National mean:         746.9
  Increase:             +192.5


In [42]:
# ---- Counter Churn comparison ----
print('\n' + '-' * 50)
print('COUNTER CHURN COMPARISON')
print('-' * 50)
for yr in ['11', '21']:
    co_ldn = f'Counter_Churn_{yr}'
    co_nat = f'Counter_Churn_nat_{yr}'
    if co_ldn in result.columns and co_nat in result.columns:
        print(f'\n20{yr}:')
        print(f'  London-only mean: {result[co_ldn].mean():>10.1f}')
        print(f'  National mean:    {result[co_nat].mean():>10.1f}')
        print(f'  Increase:         {result[co_nat].mean() - result[co_ldn].mean():>+10.1f}')



--------------------------------------------------
COUNTER CHURN COMPARISON
--------------------------------------------------

2011:
  London-only mean:      666.0
  National mean:         818.7
  Increase:             +152.7

2021:
  London-only mean:      626.3
  National mean:         885.9
  Increase:             +259.6


In [43]:
# ---- External flow volumes ----
print('\n' + '-' * 50)
print('EXTERNAL FLOW VOLUMES')
print('-' * 50)
for yr in ['11', '21']:
    ei = f'Ext_Inflow_nat_{yr}'
    eo = f'Ext_Outflow_nat_{yr}'
    en = f'Ext_Net_nat_{yr}'
    if ei in result.columns:
        print(f'\n20{yr}:')
        print(f'  Mean inflow from outside:  {result[ei].mean():>8.1f}')
        print(f'  Mean outflow to outside:   {result[eo].mean():>8.1f}')
        print(f'  Mean net external:         {result[en].mean():>+8.1f}')


--------------------------------------------------
EXTERNAL FLOW VOLUMES
--------------------------------------------------

2011:
  Mean inflow from outside:     190.8
  Mean outflow to outside:      223.0
  Mean net external:            -32.2

2021:
  Mean inflow from outside:     180.8
  Mean outflow to outside:      360.1
  Mean net external:           -179.3


## 6. Export

In [44]:
out_path = OUTPUT_DIR / f'msoa_cascade_national_frame_20260623.csv'
result.to_csv(out_path, index=False)
print(f'\nSaved: {out_path}')
print(f'  Shape: {result.shape}')



Saved: /Users/xing/Desktop/CASA/dissertation/outputs/msoa_cascade_national_frame_20260623.csv
  Shape: (983, 98)
